In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

# =====================================================================
# 0. Load Dataset
# =====================================================================
# Replace with your actual dataset load:
# df = pd.read_csv("your_dataset.csv")

# Demonstrative synthetic dataset matching typical tabular classifications:
np.random.seed(42)
n_samples = 300
data = {
    "Age": np.random.randint(20, 70, size=n_samples).astype(float),
    "Salary": np.random.normal(50000, 15000, size=n_samples),
    "Department": np.random.choice(["Sales", "IT", "HR"], size=n_samples),
    "Remote": np.random.choice(["Yes", "No"], size=n_samples),
    "Target": np.random.choice([0, 1], size=n_samples, p=[0.4, 0.6]),
}
df = pd.DataFrame(data)

# Inject sample missing values to demonstrate preprocessing
df.loc[np.random.choice(df.index, 5), "Salary"] = np.nan
df.loc[np.random.choice(df.index, 5), "Department"] = np.nan

# =====================================================================
# 1. Exploratory Data Analysis (EDA)
# =====================================================================
target_col = "Target"

num_samples, num_features = df.drop(columns=[target_col]).shape
numerical_cols = (
    df.drop(columns=[target_col])
    .select_dtypes(include=[np.number])
    .columns.tolist()
)
categorical_cols = (
    df.drop(columns=[target_col])
    .select_dtypes(include=["object", "category"])
    .columns.tolist()
)
class_distribution = df[target_col].value_counts(dropna=False)
missing_values = df.isnull().sum()

print("--- Exploratory Data Analysis ---")
print(f"Number of samples: {num_samples}")
print(f"Number of features: {num_features}")
print(f"Numerical attributes: {numerical_cols}")
print(f"Categorical attributes: {categorical_cols}")
print("\nClass Distribution:")
print(class_distribution)
print("\nMissing Values per Feature:")
print(missing_values)

# =====================================================================
# 2. Preprocessing & Split
# =====================================================================
# Separate feature matrix (X) and target label (y)
X = df.drop(columns=[target_col])
y = df[target_col]

# Split before imputing and encoding to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# A. Impute missing values learned strictly from training data
for col in numerical_cols:
    train_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)

for col in categorical_cols:
    train_mode = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(train_mode)
    X_test[col] = X_test[col].fillna(train_mode)

# B. Consistent Label Encoding for Categorical Features
# Map each unique categorical value seen in training set to an integer
cat_encoders = {}
for col in categorical_cols:
    unique_vals = X_train[col].unique()
    mapping = {val: idx for idx, val in enumerate(unique_vals)}
    cat_encoders[col] = mapping

    # Apply mapping; unseen test values default to -1
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping).fillna(-1).astype(int)

# Target vector format
y_train = np.array(y_train)
y_test = np.array(y_test)

# =====================================================================
# 3. Model Selection Justification
# =====================================================================
"""
Justification:
- Gaussian Naive Bayes: Best suited for continuous/real-valued features (assumes normal distribution).
- Multinomial Naive Bayes: Best suited for discrete count data (e.g., word frequencies).
- Bernoulli Naive Bayes: Best suited for binary/boolean indicator variables.
- Categorical Naive Bayes: Best suited when all predictors are discrete categorical variables.

Because real-world demographic/tabular data frequently contains real-valued, continuous
features (e.g., Age, Salary), Gaussian Naive Bayes is chosen below.
"""


# =====================================================================
# 4. Custom Naive Bayes Implementation (From Scratch)
# =====================================================================
class ScratchGaussianNB:

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        self.classes = None
        self.priors = {}
        self.means = {}
        self.variances = {}

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        self.classes = np.unique(y)
        n_samples = X.shape[0]

        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / float(n_samples)
            self.means[c] = np.mean(X_c, axis=0)
            # Add var_smoothing to variance to avoid division by zero
            self.variances[c] = np.var(X_c, axis=0) + self.var_smoothing

        return self

    def _calculate_log_likelihood(self, class_val, X):
        mean = self.means[class_val]
        var = self.variances[class_val]

        # Log of Gaussian Probability Density Function:
        # log P(x|c) = -0.5 * log(2 * pi * var) - ((x - mean)^2 / (2 * var))
        log_pdf = -0.5 * np.log(2.0 * np.pi * var) - ((X - mean) ** 2) / (
            2.0 * var
        )
        return np.sum(log_pdf, axis=1)

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        posteriors = []

        for c in self.classes:
            log_prior = np.log(self.priors[c])
            log_likelihood = self._calculate_log_likelihood(c, X)
            log_posterior = log_prior + log_likelihood
            posteriors.append(log_posterior)

        # posteriors has shape (n_classes, n_samples); pick argmax per sample
        posteriors = np.vstack(posteriors)
        best_class_indices = np.argmax(posteriors, axis=0)
        return self.classes[best_class_indices]


# Alternative scratch implementations for reference depending on dataset:
class ScratchCategoricalNB:

    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y)
        self.classes = np.unique(y)
        n_samples, n_features = X.shape
        self.priors = {
            c: np.mean(y == c) for c in self.classes
        }  #[cite: 3, 4]
        self.cond_probs = {}

        for c in self.classes:
            X_c = X[y == c]
            self.cond_probs[c] = []
            for j in range(n_features):
                unique_vals = np.unique(X[:, j])
                counts = {val: np.sum(X_c[:, j] == val) for val in unique_vals}
                total_count = X_c.shape[0]
                n_categories = len(unique_vals)
                # Apply Laplace smoothing
                probs = {
                    val: (counts[val] + self.alpha)
                    / (total_count + self.alpha * n_categories)
                    for val in unique_vals
                }
                self.cond_probs[c].append(probs)

    def predict(self, X):
        X = np.asarray(X, dtype=int)
        preds = []
        for row in X:
            posteriors = {}
            for c in self.classes:
                log_prob = np.log(self.priors[c])
                for j, val in enumerate(row):
                    feature_probs = self.cond_probs[c][j]
                    log_prob += np.log(
                        feature_probs.get(val, 1e-6)
                    )  # small fallback for unseen
                posteriors[c] = log_prob
            preds.append(max(posteriors, key=posteriors.get))
        return np.array(preds)


# =====================================================================
# 5. Training & Prediction
# =====================================================================
model = ScratchGaussianNB()
model.fit(X_train.to_numpy(), y_train)

y_pred = model.predict(X_test.to_numpy())

# =====================================================================
# 6. Evaluation & Results Interpretation
# =====================================================================
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("\n--- Classifier Performance Metrics ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


# ==============================================================================
# 1. MULTINOMIAL NAIVE BAYES (FROM SCRATCH)
# ==============================================================================
class ScratchMultinomialNB:
    """
    Multinomial Naive Bayes for discrete integer count data.
    P(x | c) proportional to product of (p_cj ^ x_j)
    """
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace / Lidstone smoothing parameter
        self.classes = None
        self.class_priors = {}
        self.feature_log_probs = {}

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes = np.unique(y)
        n_samples, n_features = X.shape

        for c in self.classes:
            X_c = X[y == c]
            # Prior: P(c) = N_c / N
            self.class_priors[c] = X_c.shape[0] / float(n_samples)

            # Total counts per feature within class c
            count_per_feature = np.sum(X_c, axis=0)
            total_tokens_in_class = np.sum(count_per_feature)

            # Parameter Estimation with Laplace smoothing:
            # p_cj = (count_cj + alpha) / (total_tokens_in_class + alpha * n_features)
            smoothed_probs = (count_per_feature + self.alpha) / (
                total_tokens_in_class + self.alpha * n_features
            )
            self.feature_log_probs[c] = np.log(smoothed_probs)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        log_posteriors = []

        for c in self.classes:
            # log P(c) + sum_j [ x_j * log(p_cj) ]
            log_prior = np.log(self.class_priors[c])
            log_likelihood = X @ self.feature_log_probs[c]  # Matrix dot product
            log_posteriors.append(log_prior + log_likelihood)

        # Pick class index with max posterior probability
        log_posteriors = np.vstack(log_posteriors)
        best_indices = np.argmax(log_posteriors, axis=0)
        return self.classes[best_indices]


# ==============================================================================
# 2. BERNOULLI / BINOMIAL NAIVE BAYES (FROM SCRATCH)
# ==============================================================================
class ScratchBernoulliNB:
    """
    Bernoulli / Binomial Naive Bayes for binary 0/1 indicator features.
    P(x | c) = product of [ (p_cj ^ x_j) * ((1 - p_cj) ^ (1 - x_j)) ]
    """
    def __init__(self, alpha=1.0, binarize=None):
        self.alpha = alpha
        self.binarize = binarize
        self.classes = None
        self.class_priors = {}
        self.feature_log_probs = {}
        self.feature_neg_log_probs = {}

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        if self.binarize is not None:
            X = (X > self.binarize).astype(np.float64)

        y = np.asarray(y)
        self.classes = np.unique(y)
        n_samples = X.shape[0]

        for c in self.classes:
            X_c = X[y == c]
            N_c = X_c.shape[0]

            self.class_priors[c] = N_c / float(n_samples)

            # Parameter Estimation with Laplace smoothing:
            # p_cj = (sum(x_ij) + alpha) / (N_c + 2 * alpha)
            p_c = (np.sum(X_c, axis=0) + self.alpha) / (N_c + 2.0 * self.alpha)

            self.feature_log_probs[c] = np.log(p_c)
            self.feature_neg_log_probs[c] = np.log(1.0 - p_c)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.binarize is not None:
            X = (X > self.binarize).astype(np.float64)

        log_posteriors = []

        for c in self.classes:
            log_prior = np.log(self.class_priors[c])
            # log likelihood = sum [ x_j * log(p_j) + (1 - x_j) * log(1 - p_j) ]
            log_likelihood = (
                X @ self.feature_log_probs[c]
                + (1.0 - X) @ self.feature_neg_log_probs[c]
            )
            log_posteriors.append(log_prior + log_likelihood)

        log_posteriors = np.vstack(log_posteriors)
        best_indices = np.argmax(log_posteriors, axis=0)
        return self.classes[best_indices]